# MASFIN: End to end multi-agent portfolio pipeline

This file is the script of the full MASFIN system described in the paper and repo. It runs five crews in sequence: Postmortem (failure patterns), Screening (universe filtering with bias controls), Analysis (metrics and forecasts), Timing (entry and risk gates), and Portfolio (weights and rationale).

- Data basis: prices from Yahoo Finance and market headlines from Finnhub
- Outputs: weekly shortlist and final weighted portfolio with plain text rationale

# Environment setup - see requirements.txt

Installs core MASFIN dependencies for all crews so the notebook runs in a clean environment. Ensures consistent versions and adds optional finance tools. Prefer a requirements.txt in production and load secrets from environment variables.

### Postmortem Crew Overview

Transforms delisted and near-delisted headline history into failure patterns, bias traps, and risk signals to guide the Screening Crew while reducing survivorship and data snooping bias.

**Inputs**
- Finnhub headlines for delisted and near-delisted tickers (`news_data_delisted`)
- Env vars: `OPENAI_API_KEY`, `FINNHUB_API_KEY`

**Agents**
- Delisted Historian, Failure Pattern Analyst, Confirmation Sentinel, News Sentiment Analyst, Summary Analyst

**Outputs**
- Briefing with risk factors, failure patterns, bias checks, and citations
- Decisions documented for audit and reproducibility

In [ ]:
import os
import requests
import time
from datetime import datetime, timedelta
from crewai import Agent, Task, Crew, Process
os.environ["OPENAI_API_KEY"] = # Replace with your actual key
os.environ["LITELLM_API_KEY"] = os.environ["OPENAI_API_KEY"]
os.environ["LITELLM_MODEL"] = "gpt-4.1-nano"
os.environ["FINNHUB_API_KEY"] = # Replace with your actual key

# === Finnhub Integration ===

def get_finnhub_news(ticker, start, end):
    token = os.getenv("FINNHUB_API_KEY")
    url = (
        f"https://finnhub.io/api/v1/company-news?symbol={ticker}"
        f"&from={start}&to={end}&token={token}"
    )
    res = requests.get(url)
    if res.status_code != 200:
        return f"Error fetching news for {ticker}"
    articles = res.json()[:5]
    if not articles:
        return f"No recent news for {ticker} between {start} and {end}"
    return "\n".join([f"- {a['headline']}" for a in articles])

def collect_delisted_news(symbols, start, end):
    news_blocks = []
    for sym in symbols:
        headlines = get_finnhub_news(sym, start, end)
        if headlines and "No recent news" not in headlines:
            news_blocks.append(f"### {sym}\n{headlines}")
    return "\n\n".join(news_blocks)

# Simulated delisted or near-delisted tickers
delisted_symbols = [
    #  EV / Auto
    "NKLA",   # Nikola - Non-compliance with listing requirements, bankrupt
    "RIDE",   # Lordstown Motors - Bankrupt, failure to meet listing requirements
    "ZEV",    # Lightning eMotors - Low stock price and insufficient global market capitalization

    #  Biotech / Pharma
    "ADMP",   # Adamis Pharma - Merged, delisted
    "SBBP",   # Strongbridge - Acquired by Xeris, delisted 
    "CNSP",   # CNS Pharma - Failure to comply with minimum equity requirement
    "BLUE",   # Bluebird Bio - Failure to comply with NASDAQ listing requirements

    #  Consumer / Retail
    "BBBYQ",  # Bed Bath & Beyond - Bankruptcy (Chapter 11), delisted
    "REV",    # Revlon - Bankruptcy (Chapter 11)
    "GNLN",   # Greenlane Holdings - Failure to comply with NASDAQ listing requirements

    #  Cannabis
    "AGFY",   # Agrify - Delisting threats for failure to meet listing requirements
    "HEXO",   # HEXO Corp - Failure to comply with NYSE $1 minimum stock price

    #  Tech / Media
    "FRSX",   # Foresight Auto - Failure to comply with NASDAQ $1 minimum stock price
    "GPRO",   # GoPro - Failure to comply with NASDAQ $1 minimum stock price

    #  Fintech / Financials
    "SIEB",   # Siebert Financial - Warning for failure to comply with NASDAQ filing requirements
    "MOGO",   # Mogo Inc - Failure to comply with NASDAQ $1 minimum stock price

    #  Energy / Mining
    "HYMC",   # Hycroft Mining - Notice of failure to comply with NASDAQ $1 minumum stock price
    "CEI",    # Camber Energy - Notice of failure to satisfy NYSE listing rules
]

news_data_delisted = collect_delisted_news(
    delisted_symbols,
    start="2020-01-01", #Beginning of the year of the oldest delisting/ near delisted company we examine
    end="YYY-MM-DD" #Today; start of the simulation week
)
print(f"{len(news_data_delisted)} characters in headline") #Check

postmortem_context = (
    "**Important:** You will work with two types of information:\n"
    "- **Primary Data:** Real-world news headlines from the Finnhub API, reflecting actual market events, risks, and patterns of delisted or near-delisted stocks.\n"
    "- **Secondary Context:** Optional reference sources (Quantified Strategies, LinkedIn) that explain failure patterns or psychological biases.\n"
    "Focus your analysis on the *Primary Data*. Use Secondary Context only to define or support what you observe in the headlines.\n"
    "Do not begin your reasoning from theory; begin from the data, and use theory only if it adds value or clarity.\n"
    "You are the first crew in a five-crew system that aims to produce a small stock portfolio that outperforms the market in the short run. Your job is to meet the criteria here and create a finalized report with all findings to send to the screening crew.\n"
    "1. Goal Alignment: Understand failure by analyzing patterns in delisted, near-delisted, or failed stocks. This helps prevent survivorship bias (and other forms of bias) and adds caution to screening.\n"
    "2. Optional Source 1: https://www.quantifiedstrategies.com/survivorship-bias-in-backtesting/\n"
    "3. Optional Source 2: https://www.linkedin.com/pulse/how-overcome-survivorship-bias-backtesting-trading-ayodeji-m-olumofe-bfl0e/\n"
    "4. Data Snooping Prevention Source: https://www.wallstreetmojo.com/data-snooping-bias/\n"
)


delisted_historian = Agent(
    role="Delisted Historian",
    goal=(
        "Surface systemic risks and patterns from companies that failed or were delisted/near-delisted, "
        "to prevent survivorship and data snooping bias in future investment screening."
    ),
    backstory=(
        "You are a postmortem financial historian whose specialty is reconstructing why companies disappear from the market.\n"
        "You look beneath headlines to extract hidden patterns, warning signs, and structural risk factors from the fall of real firms.\n"
        "Start your analysis from the **actual news headlines** provided (Primary Data). You may optionally reference concepts like survivorship bias or risk taxonomies (Secondary Context), but only to help explain what you're observing in the data.\n"
        "Structure your findings clearly:\n"
        "1. Brief summary of themes or trends found in the headlines\n"
        "2. List of risk factors with supporting evidence\n"
        "3. Explanation of how these risks might escape detection in standard screening models\n"
        "4. Optional: Use citations or quotes only if they clarify the real-world data\n"
        "5. Free to make your own judgement calls\n\n"
        + postmortem_context
    ),
    verbose=True
)


failure_pattern_analyst = Agent(
    role="Failure Pattern Analyst",
    goal=(
        "Identify recurring signs of corporate collapse (e.g. debt overload, fraud, leadership breakdown) "
        "using real-world news from delisted/near-delisted stocks."
    ),
    backstory=(
        "You are a forensic analyst trained to identify warning signs in failing or near-failure companies. "
        "Your job is to help build smarter screening rules by uncovering patterns that precede collapse.\n\n"
        "**Your primary input is news headlines from the Finnhub API**, covering multiple delisted/near-delisted companies. "
        "Do not generalize based on theory; extract your conclusions from the data.\n\n"
        "You may optionally cite secondary sources like HBR or Investopedia, but only to support your observations from the headlines.\n\n"
        "Structure your output as follows:\n"
        "1. List of failure pattern themes (e.g. cash flow crisis, leadership turnover, product recalls)\n"
        "2. Concrete examples from the news headlines (quotes or paraphrases)\n"
        "3. Explanation of why these signals matter in early screening\n"
        "4. Free to make your own judgement calls\n\n"
        + postmortem_context
    ),
    verbose=True
)


confirmation_sentinel = Agent(
    role="Confirmation Sentinel",
    goal="Detect signs of confirmation bias and inject counterexamples to protect against overconfident screening assumptions.",
    backstory=(
        "You are a behavioral finance watchdog tasked with enforcing objectivity. "
        "Your job is to challenge assumptions, detect cognitive bias, and introduce overlooked signals from the real world.\n\n"
        "**Use headlines from delisted/near-delisted companies (Finnhub API) as your primary source of evidence.** "
        "Look for patterns where risk signals may have been ignored, dismissed, or underweighted; especially by over-optimistic market narratives.\n\n"
        "You may reference psychology and behavioral economics (e.g. confirmation bias) as a tool to explain what's happening, "
        "but your insights should be grounded in the observed data.\n\n"
        "Structure your output as follows:\n"
        "1. Define confirmation bias in your own words\n"
        "2. Identify at least 2–3 examples of bias from the Finnhub news headlines\n"
        "3. Suggest a checklist or critical questions to help future analysts spot bias\n"
        "4. Clearly explain how this protects the Postmortem Crew from flawed reasoning\n"
        "5. Free to make your own judgement calls\n\n"
        + postmortem_context
    ),
    verbose=True
)


news_sentiment_analyst = Agent(
    role="News Sentiment Analyst",
    goal="Analyze aggregated news headlines from delisted/near-delisted companies to uncover sentiment trends, surface systemic risks, and validate or challenge failure patterns found by other agents.",
    backstory=(
        "You are a market sentiment analyst focused on reverse-engineering risk themes from real news data. "
        "You process headline clusters from multiple delisted/near-delisted companies to extract broader signals that may indicate structural weaknesses, hidden market fears, or ignored red flags.\n\n"
        "**Use the Finnhub API headlines as your primary data source.** Avoid relying on abstract sentiment theory unless it helps clarify the observed trends.\n\n"
        "Your role is both interpretive and investigative: find recurring themes, detect tone shifts, and assess how news reflects or contradicts other postmortem insights.\n\n"
        "Structure your analysis as follows:\n"
        "1. General sentiment summary (bullish, bearish, neutral)\n"
        "2. List of common themes or emotional tones (e.g. panic, denial, hype)\n"
        "3. Systemic risks inferred from those themes (e.g. overleveraging, market distrust)\n"
        "4. Comment on whether these signals align or contradict other failure patterns\n"
        "5. Free to make your own judgement calls\n\n"
        + postmortem_context
    ),
    verbose=True
)


summary_analyst = Agent(
    role="Summary Analyst",
    goal="Consolidate insights from the Postmortem Crew into a comprehensive, structured briefing for the next crew in the system.",
    backstory=(
        "You are the final synthesizer of the Postmortem Crew. You collect the findings from four expert agents who examined delisted/near-delisted stock data. "
        "Your role is to clearly organize and communicate:\n"
        "- What signals were consistently found?\n"
        "- What patterns of failure stood out?\n"
        "- What types of cognitive bias may mislead future analysts?\n"
        "- Where did sentiment support or contradict other findings?\n\n"
        "You work **only with the outputs** of the other agents. Do not create new analysis. Your goal is clarity, not creativity.\n\n"
        "Structure your report for handoff to the next crew, who will be building the actual portfolio filter. Keep it clean, structured, and high-signal.\n\n"
        + postmortem_context
    ),
    verbose=True
)


delisted_task = Task(
    description=(
        f"The following headlines were pulled from real news stories about delisted/near-delisted companies:\n\n{news_data_delisted}\n\n"
        "Your task:\n"
        "1. Identify **risk factors or red flags** that appear repeatedly in the headlines (e.g. debt issues, legal trouble, leadership exits).\n"
        "2. Explain how ignoring these delisted/near-delisted firms could lead to **survivorship bias** in screening models.\n"
        "3. Describe how analyzing failures also helps avoid **data snooping bias**, by anchoring in real-world outcomes.\n"
        "4. Use **Investopedia** or other sources only if it helps support what you're seeing in the **data**; do not lead with theory.\n"
        "5. Free to make your own judgement calls.\n\n"
        "Structure your output as follows:\n"
        "- List of risk themes found\n"
        "- Explanation of survivorship and data snooping bias prevention\n"
        "- Optional: Quotes or citations that enhance the insight"
    ),
    expected_output="A structured summary of failure risk factors with logic and real-world grounding, with optional citations.",
    agent=delisted_historian
)


failure_pattern_task = Task(
    description=(
        f"Below is a collection of real-world news headlines from multiple delisted/near-delisted companies:\n\n{news_data_delisted}\n\n"
        "Your job:\n"
        "1. Identify recurring **failure signals** in the headlines, such as fraud, debt crisis, regulatory action, or leadership instability.\n"
        "2. Categorize these signals into **systemic risk types** (e.g. financial risk, governance breakdown, market rejection).\n"
        "3. Explain how these patterns, if detected early, could help reduce **confirmation bias** in screening; by surfacing overlooked or contradictory data.\n"
        "4. You may use academic or expert sources (e.g. https://hbr.org/2011/07/why-companies-fail) to clarify or support what you observe, but do not lead with theory.\n"
        "5. Free to make your own judgement calls.\n\n"
        "Structure your output clearly:\n"
        "- Failure pattern theme → News-based example(s)\n"
        "- Systemic risk category\n"
        "- How this signal helps challenge biased assumptions in screening\n"
        "- Optional: Quote or reference to reinforce insight"
    ),
    expected_output="A structured list of failure patterns with examples and bias-reduction logic, with optional citations.",
    agent=failure_pattern_analyst
)


confirmation_bias_task = Task(
    description=(
        f"Below are real-world headlines from delisted/near-delisted companies:\n\n{news_data_delisted}\n\n"
        "Your job is to evaluate this data for signs of **confirmation bias in investing.**\n"
        "1. Define confirmation bias in simple terms, specifically in the context of financial decision-making.\n"
        "2. Analyze how this bias may have caused investors to ignore or rationalize negative signals in the headlines.\n"
        "3. Develop a practical checklist or set of critical questions that future analysts can use to guard against this bias.\n"
        "4. Explain how your checklist contributes to the Postmortem Crew’s mission of building an unbiased stock screening system.\n"
        "5. You may draw from psychology or behavioral finance sources to support your argument; begin from the data.\n"
        "6. Free to make your own judgement calls.\n\n"
        "Structure your output as follows:\n"
        "- Definition of confirmation bias (in your own words)\n"
        "- Examples from the headlines where this bias may have been active\n"
        "- Checklist for bias mitigation (clear, practical items)\n"
        "- Explanation of how the checklist supports objective screening\n"
        "- Optional: Behavioral references or quotes that strengthen your argument"
    ),
    expected_output="A clear, structured breakdown of confirmation bias risks with real examples, checklist, and reasoning support.",
    agent=confirmation_sentinel
)


news_analysis_task = Task(
    description=(
        f"The following headlines were pulled from multiple delisted/near-delisted companies:\n\n{news_data_delisted}\n\n"
        "Your task is to analyze this aggregated news set to uncover market sentiment patterns and systemic risk signals.\n\n"
        "1. Determine the overall **sentiment tone** reflected in the headlines: bearish, bullish, or neutral.\n"
        "2. Identify and summarize any **recurring risk signals**, such as liquidity issues, regulatory actions, or operational instability.\n"
        "3. Explain how recognizing these trends can help **reduce survivorship bias** in screening models; by factoring in signals from companies that failed or near-failed.\n"
        "4. Clearly note whether these insights **reinforce or challenge** the findings of other agents (Delisted Historian, Failure Pattern Analyst, Confirmation Sentinel).\n"
        "5. Free to make your own judgement calls.\n\n"
        "Structure your output as follows:\n"
        "- Sentiment summary (label + 1–2 sentence rationale)\n"
        "- List of risk signals or themes with brief explanations\n"
        "- Survivorship bias prevention insight\n"
        "- Cross-agent alignment (who your insights support or contradict)\n"
        "- Optional: Citations or signal examples from headlines"
    ),
    expected_output="A structured sentiment and trend analysis tied to risk prevention and cross-agent alignment.",
    agent=news_sentiment_analyst
)


summary_task = Task(
    description=(
        "You are tasked with summarizing and synthesizing the outputs from the full Postmortem Crew.\n\n"
        "Use only what the following agents produced:\n"
        "- Delisted Historian (risk factors from delisted firms or near-delisted)\n"
        "- Failure Pattern Analyst (collapse signals and systemic risks)\n"
        "- Confirmation Sentinel (bias checklist)\n"
        "- News Sentiment Analyst (headline sentiment and recurring themes)\n\n"
        "**Do not generate new analysis.** Only organize and explain what the above agents found, "
        "so it’s ready for the next crew.\n\n"
        "Structure your output with clear sections:\n"
        "1.  Delisting or near-delisted Risk Factors (bulleted list)\n"
        "2.  Corporate Failure Patterns (themes + brief rationale)\n"
        "3.  Confirmation Bias Checklist (practical steps)\n"
        "4.  Sentiment & Theme Summary (tone, risk signals, agreement/disagreement)\n"
        "5.  Postmortem Context Coverage (How items 1–5 from the context were satisfied)\n"
        "6.  References & Quotes (if any agents included them — cite them cleanly)\n\n"
        "This report will be handed to the next crew. Make it concise, structured, and high-signal."
    ),
    expected_output="Structured multi-section summary (bullets, logic, references).",
    agent=summary_analyst 
)

postmortem_crew = Crew(
    agents=[
        delisted_historian,
        failure_pattern_analyst,
        confirmation_sentinel,
        news_sentiment_analyst,
        summary_analyst
    ],
    tasks=[
        delisted_task,
        failure_pattern_task,
        confirmation_bias_task,
        news_analysis_task,
        summary_task
    ],
    process=Process.sequential
)

print("Running Postmortem Crew...\n")
results = []
for task in postmortem_crew.tasks:
    print(f"\n Executing task for agent: {task.agent.role}...")
    result = task.execute_sync()
    print(f" Result from {task.agent.role}:\n{result}\n{'-'*60}")
    results.append({
        "agent": task.agent.role,
        "output": result
    })
    time.sleep(25)
print("\n Final Postmortem Crew Report (Complete):")
for res in results:
    print(f"\n Agent: {res['agent']}")
    print(res['output'])
    print("="*80)

# Market Headlines Helper

Fetches the most recent general market headlines from Finnhub and formats them as a bulleted string for `news_data`. Used to ground sentiment and macro tasks.

- Requires `FINNHUB_API_KEY` in environment
- Do not commit secrets
- Prints up to 100 headlines for quick inspection

In [ ]:
import os
import requests
from datetime import datetime, timezone
from typing import List, Dict

# Set your Finnhub API key for this session (do not commit secrets!)
os.environ["FINNHUB_API_KEY"] = #Your key  here

FINNHUB_BASE = "https://finnhub.io/api/v1"


def _require_api_key() -> str:
    api_key = os.getenv("FINNHUB_API_KEY")
    if not api_key:
        raise ValueError("FINNHUB_API_KEY not set in environment variables")
    return api_key


def _fmt_headlines(items: List[Dict], limit: int) -> str:
    if not items:
        return "No headlines found."
    lines = []
    for art in items[:limit]:
        dt = datetime.fromtimestamp(art.get("datetime", 0), tz=timezone.utc).strftime("%Y-%m-%d")
        source = art.get("source") or "Unknown"
        headline = (art.get("headline") or "").strip().replace("\n", " ")
        lines.append(f"- {dt} | {source} | {headline}")
    return "\n".join(lines)


def get_market_news(limit: int = 100, timeout: int = 10) -> str:
    """
    Fetch the most recent market-wide headlines from Finnhub /news.
    Returns up to `limit` items as a bulleted string.
    """
    api_key = _require_api_key()
    resp = requests.get(
        f"{FINNHUB_BASE}/news",
        params={"category": "general", "token": api_key},
        timeout=timeout,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Error fetching news: {resp.status_code} - {resp.text}")

    items = resp.json() or []
    return _fmt_headlines(items, limit)


if __name__ == "__main__":
    print("Most recent 100 market headlines:")
    print(get_market_news())

### Screening Crew Overview

Converts postmortem insights and current news into a reproducible shortlist for downstream analysis. Applies liquidity and data quality filters, fundamentals checks, and explicit bias controls.

**Inputs**
- Postmortem summary
- Most recent 100 general market headlines (`news_data`)
- Basic prices and fundamentals

**Agents**
- Sentiment Analyst, Market Trend Analyst, Stock Screener, Fundamental Checker, Screening Summary Analyst

**Outputs**
- Shortlist of 50-100 tickers with rationale and risk flags
- Decisions documented for audit and reproducibility

In [ ]:
news_data = get_market_news(limit=100) #100 recent articles that represent the overall market
print(f"{len(news_data)} characters in headline")
screening_context = (
    "You are the **second crew** in a multi-agent investment pipeline. Your goal is to create a high-integrity shortlist of 50-100 stocks that can outperform the market in the short run.\n\n"
    """
    COPY & PASTE POSTMORTEM SUMMARY ANALYST OUTPUT HERE
    """
    
    "#### 2.  Screening Instructions:\n"
    "Use the following as implementation logic, not theory:\n"
    "1. Sector-specific filters - align with risk by industry\n"
    "2. Financial health - Altman Z, debt/equity, cash burn\n"
    "3. Leadership - flag firms with executive churn or governance concerns\n"
    "4. Innovation gaps - penalize stagnant product pipelines\n"
    "5. Legal history - score risk for litigation, fines, or delisting events\n"
    "6. Sentiment signals - apply NLP on headlines for tone and trust\n"
    "7. Macro stress testing - simulate exposure to recessionary scenarios\n"
    "8. Analysts may document additional heuristics with justification\n\n"
    
    "#### 3.  Bias Avoidance Protocols:\n"
    "**[Survivorship Bias](https://www.investopedia.com/terms/s/survivorshipbias.asp):**\n"
    "- Include delisted stocks in training sets and signal logic\n\n"
    "**[Data Snooping Bias](https://www.wallstreetmojo.com/data-snooping-bias/):**\n"
    "- Do not adjust filters based on final performance\n"
    "- [Prevent leakage](https://machinelearningmastery.com/data-leakage-machine-learning/) between training and outcome data\n\n"
    "**[Confirmation Bias](https://www.psychologytoday.com/us/basics/confirmation-bias):**\n"
    "- Seek disconfirming evidence actively\n"
    "- Record rationale and maintain peer review on selections\n\n"
    
    "#### 4.  Final Crew Objective:\n"
    "Generate a shortlist of 50-100 stocks:\n"
    "- High short-term upside\n"
    "- Low structural risk (per Postmortem signals)\n"
    "- Compliant with all bias protocols\n"
    "- Sentiment-aligned but not overhyped\n\n"
    
    "#### 5.  Reference Materials (secondary):\n"
    "- [Investopedia: Survivorship Bias](https://www.investopedia.com/terms/s/survivorshipbias.asp)\n"
    "- [Wall Street Mojo: Data Snooping Bias](https://www.wallstreetmojo.com/data-snooping-bias/)\n"
    "- [HBR: Why Companies Fail](https://hbr.org/2011/07/why-companies-fail)\n"
    "- [Psychology Today: Confirmation Bias](https://www.psychologytoday.com/us/basics/confirmation-bias)\n"
    "- [CFA Institute: Behavioral Finance](https://www.cfainstitute.org/en/research/foundation/2020/behavioral-finance)\n"
    "- [MachineLearningMastery: Data Leakage](https://machinelearningmastery.com/data-leakage-machine-learning/)\n"
    "- [Investopedia: Fundamental Analysis](https://www.investopedia.com/terms/f/fundamentalanalysis.asp)\n"
)


sentiment_analyst = Agent(
    role="Sentiment Analyst",
    goal="Assess overall investor sentiment across the market using news data.",
    backstory=(
        "You analyze news tone across the entire market. Your goal is to identify investor mood, fear patterns, and tone shifts "
        "based on real headlines retrieved from the Finnhub API. Avoid focusing on specific tickers - instead, highlight macro themes such as inflation fears, "
        "technology sector selloffs, or Federal Reserve policy anxiety.\n\n"
        "Your analysis should be grounded in the most recent 100 general-market news articles to capture the current sentiment environment.\n"
        "You may document additional heuristics if necessary, provided they are justified and reproducible.\n\n"
        f"{screening_context}"
    ),
    verbose=True
)


market_trend_analyst = Agent(
    role="Market Trend Analyst",
    goal="Identify macroeconomic forces and sector momentum patterns.",
    backstory=(
        "You observe capital flow, sector strength, and macro cycles. From interest rate shifts to geopolitical risks, your insights inform which types of stocks are "
        "likely to benefit in the current regime.\n\n"
        "Do not produce a list of tickers - instead, explain which areas of the market are gaining traction.\n"
        "Base your reasoning on patterns reflected in the most recent 100 general-market news articles, as these capture the latest macroeconomic and sector themes.\n"
        "You may apply your own judgment when interpreting ambiguous signals, provided the rationale is clear and reproducible.\n\n"
        f"{screening_context}"
    ),
    verbose=True
)


screener_agent = Agent(
    role="Stock Screener",
    goal="Apply minimum liquidity and data completeness standards.",
    backstory=(
        "Your role is to apply strict filters that remove illiquid, data-deficient, or structurally unfit stocks from consideration. "
        "You do not recommend stocks - you only define and explain the pre-filtering logic you would apply to a large universe.\n\n"
        "Your reasoning should take into account insights from the most recent 100 general-market news articles, which provide context on "
        "prevailing liquidity conditions, reporting reliability, and structural risks in the market.\n"
        "You may apply your own judgment when interpreting ambiguous cases, provided the rationale is clear and reproducible.\n\n"
        f"{screening_context}"
    ),
    verbose=True
)


fundamental_checker_agent = Agent(
    role="Fundamental Checker",
    goal="Disqualify weak or overvalued stocks from screening output.",
    backstory=(
        "You do not generate new tickers. Your role is to explain which financial fundamentals (e.g., earnings strength, valuation health, margin stability) "
        "are required to pass the screening gate.\n\n"
        "Your analysis should be informed by patterns observed in the most recent 100 general-market news articles, particularly regarding earnings sentiment, "
        "valuation pressures, and macro-level profitability trends.\n"
        "You may apply your own judgment when interpreting borderline fundamentals, provided the rationale is clear and reproducible.\n\n"
        f"{screening_context}"
    ),
    verbose=True
)


summary_agent = Agent(
    role="Screening Summary Analyst",
    goal="Consolidate all screening insights and produce the final shortlist of 50-100 tickers.",
    backstory=(
        "You receive reasoning from:\n"
        "- Sentiment Analyst (macro tone)\n"
        "- Market Trend Analyst (sector and macro themes)\n"
        "- Screener Agent (filter logic)\n"
        "- Fundamental Checker (valuation and earnings standards)\n"
        f"- Postmortem Crew insights provided in {screening_context}\n\n"
        "Your synthesis should also be informed by the most recent 100 general-market news articles, as they provide the broader sentiment and macro backdrop "
        "against which screening decisions are made.\n\n"
        "Based on all of these inputs, you generate the final shortlist of tickers. You do not invent new insights - you only consolidate and organize what has "
        "already been provided."
    ),
    verbose=True
)


sentiment_task = Task(
    description=(
        f"Market headlines (most recent 100 general-market articles):\n\n{news_data}\n\n"
        "Instructions:\n"
        "- Classify overall market tone as Bullish, Bearish, or Mixed.\n"
        "- Identify dominant sentiment themes (e.g., inflation, tech sector, Fed policy).\n"
        "- Summarize how these themes may influence stock selection logic.\n"
        "- You may apply your own judgment when interpreting ambiguous signals, provided rationale is explicit."
    ),
    expected_output="Structured sentiment analysis including overall tone, thematic breakdown, and implications for screening (no tickers).",
    agent=sentiment_analyst
)


market_trend_task = Task(
    description=(
        f"Macro trends from the most recent 100 general-market news articles:\n\n{news_data}\n\n"
        "Instructions:\n"
        "- Identify major economic tailwinds or headwinds.\n"
        "- Highlight sectors gaining or losing investor attention.\n"
        "- Suggest how these trends should inform screening priorities.\n"
        "- Explain bias-prevention logic applied during interpretation.\n"
        "- You may apply your own judgment when interpreting ambiguous signals, provided rationale is explicit.\n"
        "- No ticker recommendations allowed."
    ),
    expected_output="Structured macroeconomic and sector trend insights with bias mitigation rationale (no tickers).",
    agent=market_trend_analyst
)


screener_task = Task(
    description=(
        f"Market headlines to reference (most recent 100 general-market articles):\n\n{news_data}\n\n"
        "Instructions:\n"
        "- Define screening filters for investability.\n"
        "- Include liquidity thresholds, daily volume requirements, and penny stock exclusions.\n"
        "- Address survivorship and data quality bias explicitly.\n"
        "- Explain filtering logic only — do not return tickers.\n"
        "- You may apply your own judgment when interpreting ambiguous cases, provided rationale is explicit."
    ),
    expected_output="Structured description of pre-screening filters with clear bias mitigation rationale (no tickers).",
    agent=screener_agent
)


fundamental_task = Task(
    description=(
        f"Market headlines to reference (most recent 100 general-market articles):\n\n{news_data}\n\n"
        "Instructions:\n"
        "- Define fundamental disqualification criteria.\n"
        "- Include weak earnings or declining profit margins.\n"
        "- Exclude overvalued firms (e.g., extreme P/E, negative free cash flow, excessive leverage).\n"
        "- Explain how these fundamentals mitigate confirmation or sentiment bias.\n"
        "- Do not output any tickers — only provide the framework and logic.\n"
        "- You may apply your own judgment when interpreting ambiguous cases, provided rationale is explicit."
    ),
    expected_output="Structured description of fundamental disqualification criteria with justification and bias-mitigation rationale (no tickers).",
    agent=fundamental_checker_agent
)


summary_task = Task(
    description=(
        "Review outputs from the following:\n"
        "- Sentiment Analyst\n"
        "- Market Trend Analyst\n"
        "- Screener Agent\n"
        "- Fundamental Checker Agent\n"
        f"- Postmortem Crew insights provided in {screening_context}\n\n"
        "Assume the Sentiment and Market Trend inputs reflect the most recent 100 general-market news articles.\n\n"
        "Based on these, synthesize a final list of 50-100 stocks.\n"
        "- Include only tickers that pass all layers of reasoning.\n"
        "- Summarize key rationale from each contributing agent.\n"
        "- Explain how survivorship, data snooping, and confirmation biases were reduced.\n"
        "- Optional: bullet highlights for exceptional candidates.\n"
        "- Do not introduce new analysis beyond what was provided."
    ),
    expected_output="Final shortlist of 50-100 tickers with per-ticker rationale, cross-agent justification, and explicit bias-mitigation notes.",
    agent=summary_agent
)


screening_crew = Crew(
    agents=[
        sentiment_analyst,
        market_trend_analyst,
        screener_agent,
        fundamental_checker_agent,
        summary_agent
    ],
    tasks=[
        sentiment_task,
        market_trend_task,
        screener_task,
        fundamental_task,
        summary_task
    ],
    process=Process.sequential
)

# ===  Run Crew
print(" Running Screening Crew...\n")
results = []
for task in screening_crew.tasks:
    print(f" Executing: {task.agent.role}")
    result = task.execute_sync()
    print(f" Result from {task.agent.role}:\n{result}\n{'='*60}")
    results.append(result)
    time.sleep(20)

print("\n Final Output:")
for r in results:
    print(r)

In [ ]:
#Screening Stock List
#update output
screening_stock_list = [] #Update with 50-100 tickers outputted by the Screening Crew; tip copy and paste the output into AI and tell it to give you the tickers in a python list

last_week_stocks= [] #Update with tickers of last week's final portfolio


### Beginning and End Data Snapshots

MASFIN produces **performance snapshots** at both the **start** and **end** of each trading week.  
These are calculated in `Calculations.py` (available on our GitHub) to ensure consistency and reproducibility across all crews.

**Purpose**  
- Capture a standardized set of financial metrics for all tickers in `merged_tickers` (the union of screened tickers and prior-week holdings).  
- Provide a baseline at the week’s start (`beginning_data`) and an outcome set at the week’s end (`end_data`).  
- Include metrics such as 21-Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum, Beta, Alpha, 5-Day Return, Recent Z-Score, Volume Trends, Residual Volume, and Price Relative to 5-Day MA.  
- The **end snapshot** also adds a percentage change column to track week-over-week performance.

**Usage by Agents**  
- **Analysis Crew**: Uses beginning and end data to evaluate short-term performance. 

Together, these snapshots provide a reproducible reference point for performance tracking and agent decision-making throughout MASFIN.

In [ ]:
#Merge both sets (for unified evaluation this week)
merged_tickers = screening_stock_list + last_week_stocks

#Performance snapshot from the start of the week
#Calculate for all tickers in merged_tickers using Calculations.py on our GitHub; below is an example of how one could format the data
beginning_data = """Begin or End | Company Ticker | 21 Day Return | Volatility | Sharpe Ratio | Max Drawdown | Momentum   | Beta     | Alpha     | 5-Day Return | Recent Z-Score (Return) | 5-Day Volume Trend | Residual Volume | Price Relative to 5-Day MA
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Beginning     | AAPL           | -0.036056     | 0.135812   | -3.517919     | -0.056063    | -7.569992   | 1.146742 | -0.001988 | -0.0538      | -1.6676                 | 16243970           | 7963540          | -0.031
              | NVDA           | 0.097826      | 0.258475   | 4.922738      | -0.034509    | 15.479996   | 1.168380 | 0.004955  | 0.0013       | -1.3871                 | 19661890           | -13720300        | -0.0164
              | GOOGL          | 0.069800      | 0.188068   | 4.855925      | -0.037653    | 12.340012   | 0.794241 | 0.003560  | -0.0210      | -0.7380                 | 332540             | -5459980         | -0.021
              | MSFT           | 0.053022      | 0.181769   | 3.859807      | -0.017601    | 26.389984   | 0.560684 | 0.002739  | 0.0202       | -1.1526                 | 6448730            | -11470360        | 0.0095
              | AMZN           | -0.039021     | 0.350566   | -1.331090     | -0.082696    | -8.720001   | 2.866438 | -0.002082 | -0.0721      | -1.9411                 | 26255850           | 5821960          | -0.0605
              | AVGO           | 0.052739      | 0.272529   | 2.634427      | -0.046196    | 14.460022   | 1.430059 | 0.002734  | -0.0053      | -0.8659                 | 2176710            | -2235260         | -0.0227
              | PFE            | -0.053276     | 0.237291   | -2.943517     | -0.080998    | -1.321865   | 0.290915 | -0.002795 | -0.0524      | 1.5527                  | 6474070            | -5484820         | -0.0147
              | TMO            | 0.105899      | 0.469318   | 3.067565      | -0.068804    | 44.339996   | 2.308820 | 0.005528  | -0.0319      | -0.3138                 | -106090            | -97060           | -0.0252
              | WMT            | -0.008656     | 0.148157   | -0.707506     | -0.049824    | -0.860001   | -0.181879| -0.000401 | 0.0105       | 0.5999                  | 1053160            | 125440           | 0.0050
              | UNH            | -0.217115     | 0.451978   | -6.916671     | -0.227267    | -65.939987  | 2.658340 | -0.012619 | -0.1540      | -0.4071                 | 3632050            | 2834440          | -0.0831
              | NFLX           | -0.101596     | 0.293794   | -4.681616     | -0.101596    | -131.020020 | 0.262124 | -0.005479 | -0.0185      | 0.2735                  | 184840             | 176100           | -0.0090
              | RMD            | 0.088881      | 0.197502   | 5.824217      | -0.023482    | 22.790009   | -0.032602| 0.004567  | 0.0147       | 1.4485                  | 150420             | 2360             | 0.0118
              | DHR            | -0.012724     | 0.326524   | -0.365278     | -0.084862    | -2.540009   | 1.389031 | -0.000585 | -0.0408      | 0.4495                  | 224000             | -236220          | -0.0224
              | CSCO           | -0.026404     | 0.122505   | -2.836920     | -0.031182    | -1.820000   | 1.078116 | -0.001466 | -0.0230      | -1.3534                 | 545060             | 2256360          | -0.0112
              | JNJ            | 0.077671      | 0.271822   | 3.782516      | -0.028426    | 12.059998   | 0.110982 | 0.004071  | -0.0058      | 1.3485                  | 519920             | -478280          | 0.0036
              | KO             | -0.030277     | 0.161397   | -2.448335     | -0.043938    | -2.150002   | -0.494009| -0.001528 | -0.0045      | 1.0444                  | 584740             | 209760           | 0.0039
              | NKE            | -0.024957     | 0.304892   | -0.954345     | -0.059323    | -1.909996   | 1.262568 | -0.001256 | -0.0216      | 0.1365                  | -1762450           | -309480          | -0.0273
"""


#Performance snapshot from the end of the week
#Calculate for all tickers in merged_tickers using Calculations.py on our GitHub; below is an example of how one could format the data
#Make sure to include pecent change as a column here as well
end_data = #format same as above

# Analysis Crew Metrics Computation (Additional to Beginning and End Snapshots)

Builds a 30 day feature table for each ticker and the S&P 500 benchmark to support screening and analysis.

- Metrics per ticker: 21D return, annualized volatility, Sharpe ratio, max drawdown, momentum, beta and alpha vs ^GSPC (OLS), 5D return, 5D momentum, recent return z score (last 5), price vs MA(5), volume trend and residual
- Data: Yahoo Finance with auto adjusted prices; market return from ^GSPC close percent change
- Output: `metrics_df` and `Full_Metrics_Output.xlsx` for downstream crew use

Requires yfinance, pandas, numpy, scipy, statsmodels.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import zscore
import statsmodels.api as sm
def compute_all_metrics(tickers):
    end_date = date.today()
    start_date = end_date - timedelta(days=30)
    # Download 30-day daily price data
    price_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True)
    market_data = yf.download("^GSPC", start=start_date, end=end_date, auto_adjust=True)
    market_data.columns = market_data.columns.get_level_values(0)  # <- FIXED LINE
    market_data['Market Return'] = market_data['Close'].pct_change().dropna()

    # Download 10-day data for volume trends
    vol_data = yf.download(tickers, period="10d", interval="1d", group_by='ticker', auto_adjust=True)

    results = []

    for ticker in tickers:
        try:
            df = price_data[ticker].dropna()
            vol_df = vol_data[ticker].dropna()
            if df.empty or vol_df.empty or len(df) < 10:
                raise ValueError("Insufficient data")

            df['Return'] = df['Close'].pct_change()
            total_return = round((df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1, 6)
            volatility = round(df['Return'].std() * np.sqrt(252), 6)
            sharpe = round((df['Return'].mean() * 252) / (df['Return'].std() * np.sqrt(252)), 6)
            max_dd = round(((df['Close'] / df['Close'].cummax()) - 1).min(), 6)
            momentum = round(df['Close'].iloc[-1] - df['Close'].iloc[0], 6)

            # Beta/Alpha
            stock_returns = df[['Return']].dropna()
            merged = pd.merge(stock_returns, market_data[['Market Return']], left_index=True, right_index=True)
            if merged.empty:
                beta = alpha = np.nan
            else:
                X = sm.add_constant(merged['Market Return'])
                y = merged['Return']
                model = sm.OLS(y, X).fit()
                beta = round(model.params['Market Return'], 6)
                alpha = round(model.params['const'], 6)

            # 5-day Return and Momentum
            if len(df) >= 6:
                ret_5d = round((df['Close'].iloc[-1] / df['Close'].iloc[-6]) - 1, 6)
                mom_5d = round(df['Close'].iloc[-1] - df['Close'].iloc[-6], 6)
            else:
                ret_5d = mom_5d = np.nan

            # Z-score of recent returns
            returns = df['Close'].pct_change().dropna()
            if len(returns) >= 5:
                z_ret = round(zscore(returns.tail(5))[-1], 6)
            else:
                z_ret = np.nan

            # Price vs 5-day MA
            ma5 = df['Close'].rolling(5).mean()
            price_vs_ma = round((df['Close'].iloc[-1] / ma5.iloc[-1]) - 1, 6) if not pd.isna(ma5.iloc[-1]) else np.nan

            # Volume Trend and Residual
            if len(vol_df) >= 5:
                y = vol_df['Volume'].tail(5)
                X = sm.add_constant(np.arange(len(y)))
                model = sm.OLS(y, X).fit()
                vol_trend = round(model.params[1], 6)
                vol_residual = round(model.resid.iloc[-1], 6)
            else:
                vol_trend = vol_residual = np.nan

            results.append([
                ticker, total_return, volatility, sharpe, max_dd, momentum,
                beta, alpha, ret_5d, mom_5d, z_ret, price_vs_ma, vol_trend, vol_residual
            ])

        except Exception as e:
            print(f"Error with {ticker}: {e}")
            results.append([ticker] + [np.nan]*13)

    columns = [
        "Ticker", "21D Return", "Volatility", "Sharpe Ratio", "Max Drawdown", "Momentum",
        "Beta", "Alpha", "5D Return", "5D Momentum", "Z-Score (Return)",
        "Price vs MA(5)", "Volume Trend", "Residual Volume"
    ]

    return pd.DataFrame(results, columns=columns)

tickers=merged_tickers

metrics_df = compute_all_metrics(tickers)
metrics_df.to_excel("Full_Metrics_Output.xlsx", index=False)
print(metrics_df)

### Analysis Crew Overview

Purpose: Evaluate Final Screened Candidates and Prior-Week Survivors using time-valid data in the {backtest_start} to {backtest_end} window to produce a shortlist for portfolio construction.

Inputs
- `metrics_df` 30 day baselines and recent stats
- `beginning_data`, `end_data` start and end snapshots
- `news_data` Finnhub sentiment
**Agents**
- Backtesting Agent, Statistical Enricher, Forecasting Analyst, and Historical Trend Analyst

Outputs
- Bias aware shortlist of 35–50 tickers with brief rationales, ready for allocation

In [ ]:
print(f"{len(news_data)} characters in headline") #Same 100 recent headlines that reflect the overall market
analysis_context = f"""
You are the **third crew** in a multi-agent pipeline tasked with producing a short-term outperforming stock portfolio.

Your deliverable is a condensed table of all relevant statistical performance data across the target tickers.

### Inputs:
**1. Final Screened Candidates:**
{screening_stock_list}

**2. Prior-Week Survivors:**
{last_week_stocks}

**3. Last Week: Begin Performance Snapshot** (collected at market open or near market open)
{beginning_data}

**4. Last Week: End Performance Snapshot** (collected EOD Friday at 4 pm ET)
{end_data}

**5. Full Week: Percent Change per Ticker**
Also in {end_data}

### Your Objective:
- Apply statistical evaluation to all tickers using metrics like:
  21 Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum, Beta, Alpha,
  5-Day Return, Recent Z-Score (Return), 5-Day Volume Trend, Residual Volume, Price Relative to 5-Day MA, and Total Gain/Loss percent since time of purchase.
- Compare beginning vs. end-week performance to understand short-run viability and risk-adjusted returns.
- Ensure all assessments are strictly based on what was known at the time, not hindsight data.

### Required Frameworks:
- Interpret trends using logic that avoids data leakage, confirmation bias, and other biases.

### References (Optional):
- https://www.utradealgos.com/blog/what-are-the-key-metrics-to-track-in-algo-trading-backtesting/
- https://bookdown.org/palomar/portfoliooptimizationbook/8.4-backtesting-market-data.html

### Overall Structure:
Inputs:
- Final Screened Candidates
- Prior-Week Survivors
- Weekly Performance Snapshots (Start and End of Week)

    ↓

Backtesting Agent:
- Computes raw performance metrics using Yahoo Finance price data (auto-adjusted close):
  • 5-Day Return, Volatility, Sharpe Ratio, Alpha/Beta
  • Z-Scores, Max Drawdown, Regression score
  • Volume trends, Price relative to moving averages

    ↓

Parallel Evaluation (3 agents):

→ Statistical Enricher:
  - Interprets metrics and makes Keep/Drop decisions
  - Flags anomalies and risk indicators

→ Forecasting Analyst:
  - Predicts 5-day price direction (↑/↓) with confidence score
  - Justifies with Z-score, momentum, macro trends

→ Historical Trend Analyst:
  - Classifies patterns as Aligned, Deviant, or Neutral
  - Uses prior trend analogs + macro headline context

    ↓

Summary Synthesizer:
- Consolidates outputs from all agents
- Drops tickers rejected by agents
- Produces final list of 35-50 candidates
- Includes rationale and bias control summary
"""


# Backtest window and lookback (set these once for the run)
backtest_start = "YYYY-MM-DD"  # First day of simulation week
backtest_end = "YYYY-MM-DD"    # Last day of simulation week
lookback_days = 30             # matches metrics_df construction

backtesting_agent = Agent(
    role="Quantitative Backtesting Analyst",
    goal=(
        f"Objectively perform short-term backtesting using historical data from Yahoo Finance stored in metrics_df. "
        f"This simulation covers the upcoming window from {backtest_start} to {backtest_end}, using only data available from the prior {lookback_days} days."
    ),
    backstory=(
        "You are a quantitative analyst responsible for simulating near-term performance of screened tickers. "
        "Your inputs are structured market data (price, volume, risk factors) sourced from Yahoo Finance, optionally supported by headline sentiment trends from Finnhub.\n\n"
        f"Use only information that would have been available prior to {backtest_start}. "
        "Rely on the precomputed features in metrics_df; do not recompute from future data.\n\n"
        "Metrics available in metrics_df include: 21 Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum, Beta, Alpha, "
        "5 Day Return, 5 Day Momentum, Recent Z Score (Return), 5 Day Volume Trend, Residual Volume, and Price Relative to 5 Day Moving Average.\n\n"
        f"To generate forward-looking estimates for the {backtest_start} to {backtest_end} window, apply a simple baseline method: "
        "for each metric, compute the cross-sectional mean across all tickers in the dataset and use it as the baseline prediction for each stock. "
        "Document any deviations from this baseline and the rationale.\n\n"
        "You are part of the Analysis Crew, producing quantitative inputs for downstream portfolio construction. "
        "All evaluations must avoid forward looking bias and reflect real world constraints.\n\n"
        f"{analysis_context}"
    ),
    verbose=True,
    memory=True,
    allow_delegation=False
)




statistical_enricher_agent = Agent(
    role='Statistical Enricher Agent',
    goal=(
        "Interpret and contextualize statistical metrics from the backtesting agent, as well as beginning data, end data, and total gain/loss percent. "
        "Focus on extracting short-term risk/reward signals while maintaining strict data integrity."
    ),
    backstory=(
        "You are a quantitative performance analyst embedded in the Analysis Crew - the third phase of a multi-agent pipeline "
        "responsible for evaluating **Final Screened Candidates** AND **Prior-Week Survivors**. Your mission is to derive insights from statistical indicators "
        "calculated during backtesting and from metrics provided in the context (beginning data, end data, and total gain/loss percent) for the "
        f"{backtest_start} to {backtest_end} window, including:\n\n"
        "21-Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum, Beta, Alpha, 5-Day Return, "
        "Recent Z-Score (Return), 5-Day Volume Trend, Residual Volume, Price Relative to 5-Day MA, and Total Gain/Loss Percent per ticker for last week's tickers.\n\n"
        "You help turn raw quantitative outputs into meaningful inferences about short-term viability. "
        "You are highly trained in interpreting risk-adjusted performance metrics and spotting signal anomalies. "
        "Your conclusions will influence final timing and allocation decisions, so rigor and clarity are critical. "
        "Do not introduce new data; compute metrics exactly as defined in the repository to preserve reproducibility.\n\n"
        "Free to make your own judgment calls.\n\n"
        f"{analysis_context}"
    ),
    verbose=True,
    memory=True,
    allow_delegation=False,
)


forecasting_analyst = Agent(
    role='Short-Term Forecasting Analyst',
    goal=(
        f"Predict the 5-day price direction (↑ or ↓) for each ticker from Final Screened Candidates and Prior-Week Survivors "
        f"for the {backtest_start} to {backtest_end} window. "
        "Use momentum, volatility, and trend changes observed between the beginning and end of the window, along with available Finnhub sentiment. "
        "Avoid any use of future-confirmed movements or hindsight."
    ),
    backstory=(
        "You are a forecasting specialist within the Analysis Crew (3rd of 5). Your job is to assign a directional prediction "
        "to each approved ticker based on time-valid data only.\n\n"

        "You receive three key inputs:\n"
        f"1. {beginning_data} and {end_data} - quantitative snapshots from the start and end of the window, including metrics like momentum, Z-score, 5-day return, and short-term volatility.\n"
        f"2. Finnhub sentiment {news_data} for each ticker (if available).\n"
        "3. For tickers not present in beginning_data or end_data (typically Final Screened Candidates), use predictive metrics from the Backtesting Agent together with Finnhub sentiment to inform the forecast.\n\n"

        "Output a simple table with:\n"
        "- Direction (↑ or ↓)\n"
        "- Confidence (High / Medium / Low)\n"
        "- Drop? (Yes / No)\n"
        "- Brief justification (1 sentence max)\n\n"

        "Free to make judgment calls.\n\n"
        f"{analysis_context}"
    ),
    verbose=True,
    memory=True,
    allow_delegation=False
)


historical_trend_analyst = Agent(
    role='Historical Trend Analyst',
    goal=(
        f"Determine whether the recent 5-day behavior of each ticker aligns with or deviates from its 30-day historical averages "
        f"during the {backtest_start} to {backtest_end} window. "
        "Only assess tickers from Final Screened Candidates or Prior-Week Survivors."
    ),
    backstory=(
        "You are a financial pattern recognition specialist operating within the Analysis Crew (Phase 3 of MASFIN). "
        "You evaluate whether each ticker's recent price and volume behavior reflects its own historical norms.\n\n"

        "You receive the following:\n"
        f"1. {metrics_df}: 30-day baseline indicators per ticker (e.g., momentum, volatility, Z-score, return)\n"
        f"2. {beginning_data} and {end_data}: snapshots from the start and end of the window for prior-week tickers\n"
        "3. (Optional) Sentiment context via Finnhub\n\n"

        "Your approach:\n"
        "- If beginning_data and end_data are available for a ticker:\n"
        "  • Compare short-term changes (e.g., delta Z-score, delta momentum) to 30-day baselines from metrics_df.\n"
        "  • Classify the recent move as:\n"
        "    * Aligned (consistent behavior)\n"
        "    * Deviant (surprising or contradictory behavior)\n"
        "    * Neutral (inconclusive or minor variation)\n"
        "- If no recent snapshot data is available: mark as 'No comparison available'.\n\n"

        "Your task is not to forecast direction, but to determine alignment between short-term movement and historical norms. "
        "You may cite headlines or macro themes if they plausibly explain divergence. "
        "Do not introduce new data; compute metrics exactly as defined in the repository to preserve reproducibility.\n\n"

        "Free to make your own judgment calls.\n\n"
        f"{analysis_context}"
    ),
    verbose=True,
    memory=True,
    allow_delegation=False
)



summary_agent = Agent(
    role='Portfolio Decision Synthesizer',
    goal=(
        f"Compile and finalize the strongest short-term stock candidates for the {backtest_start} to {backtest_end} window "
        "from tickers that passed statistical enrichment, forecasting, and historical trend analysis. "
        "Produce a bias-aware shortlist of tickers ready for portfolio allocation."
    ),
    backstory=(
        "You are a synthesis expert in the Analysis Crew - the third phase of a multi-agent system designed to construct a high-confidence, "
        "short-term outperforming stock portfolio. Your role is to consolidate the outputs of prior agents who assessed:\n"
        "- Backtesting Simulation\n"
        "- Statistical Enrichment\n"
        "- Forecast Direction and Confidence\n"
        "- Historical Pattern Consistency\n"
        "- Drop/Keep Recommendations from each layer\n\n"

        "You work exclusively with tickers that were:\n"
        "- Screened in via the Screening Crew\n"
        "- Last week's tickers\n\n"

        "Your task is to:\n"
        "- Eliminate any ticker dropped by prior analytical stages\n"
        "- Prioritize tickers that consistently scored well across all phases\n"
        "- Avoid recency bias, confirmation bias, and overfitting to any single signal type\n\n"

        "Your final output is a shortlist of tickers marked **Final Keep**, with optional scoring or rankings if confidently supported. "
        "This list will guide portfolio construction.\n\n"
        f"{analysis_context}"
    ),
    verbose=True,
    memory=True,
    allow_delegation=False
)


backtesting_task = Task(
    description=(
        f"Perform a forward-looking short-term backtest using Yahoo Finance data stored in {metrics_df}. "
        f"Simulate performance over the {backtest_start} to {backtest_end} window using only information available up to {backtest_start}. "
        "Avoid all forms of lookahead bias.\n\n"

        "### Tickers to Evaluate\n"
        "- Include all tickers from Final Screened Candidates and Prior-Week Survivors.\n"
        "- Evaluate each ticker once, even if it appears in both lists.\n\n"

        "### Required Metrics\n"
        "21-Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum, Beta, Alpha, "
        "5-Day Return, 5-Day Momentum, Recent Z-Score (Return), 5-Day Volume Trend, Residual Volume, "
        "Price Relative to 5-Day Moving Average.\n\n"

        "### Methodology\n"
        f"- Use the global mean for each metric in {metrics_df} as the baseline prediction.\n"
        "- Supplement with Finnhub News when relevant as a methodological reference.\n"
        "- Cite credible secondary sources only when they directly inform calculations.\n"
        "- Compute all metrics exactly as defined in the repository to ensure reproducibility.\n\n"

        "### Output\n"
        "- A table of predictive values for each ticker across all required metrics.\n"
        "- Include numeric values and, if desired, visual cues (e.g., symbols or color coding).\n\n"

        "All results must be ready for downstream consumption by the next agent in the Analysis Crew."
    ),
    expected_output=(
        "A predictive table for each evaluated ticker, estimating the following metrics for the upcoming week: "
        "21-Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum, Beta, Alpha, "
        "5-Day Return, 5-Day Momentum, Recent Z-Score (Return), 5-Day Volume Trend, Residual Volume, "
        "and Price Relative to the 5-Day Moving Average."
    ),
    agent=backtesting_agent
)


statistical_enricher_task = Task(
    description=(
        f"Evaluate all tickers for the {backtest_start} to {backtest_end} window using the full set of backtested metrics from metrics_df "
        "and gain/loss data for prior holdings to decide which stocks to retain or drop before portfolio construction.\n\n"

        "### Output Requirements:\n"
        "- For each ticker, provide:\n"
        "    • Final Decision: Keep or Drop\n"
        "    • Key Supporting Metrics (e.g., Sharpe Ratio, Max Drawdown, Z-Score, Volume Trends)\n"
        "    • Brief Justification (for example, poor risk-adjusted return, extreme volatility, anomaly detection)\n\n"

        "### Methodology:\n"
        "- Use statistical judgment based on metrics_df.\n"
        f"- Optionally refer to {news_data}, Quantpedia, or similar research to support reasoning.\n"
        "- Cite sources if external research or quotes are used.\n"
        "- Compute all metrics exactly as defined in the repository to preserve reproducibility.\n\n"

        "### Bias Mitigation:\n"
        "- Avoid lookahead bias and hindsight-based decisions.\n"
        "- Explain how your process reduces confirmation bias and data mining bias.\n\n"

        "### Final Output:\n"
        "- A structured chart or table with: Ticker | Keep/Drop | Key Metrics | Justification.\n"
        "- A short paragraph explaining how bias was avoided.\n"
        "- Include citations if any external sources were referenced.\n\n"

        "Free to make your own judgment calls."
    ),
    expected_output=(
        "A structured chart showing each ticker with a Keep/Drop decision, supporting metrics, and a brief justification, "
        "plus a paragraph explaining bias controls and a list of any cited sources."
    ),
    agent=statistical_enricher_agent
)


forecasting_task = Task(
    description=(
        f"Using beginning and end of window snapshots and Finnhub sentiment, forecast the likely 5-Day price direction "
        f"for each ticker in Final Screened Candidates and Prior-Week Survivors for the {backtest_start} to {backtest_end} window.\n\n"

        "### Inputs:\n"
        f"- {beginning_data}: metrics from Monday\n"
        f"- {end_data}: metrics from Friday\n"
        f"- {news_data}: Finnhub sentiment and headline context\n"
        "- For tickers not present in beginning_data or end_data, use predictive metrics from the Backtesting Agent together with Finnhub sentiment.\n\n"

        "### Output Table:\n"
        "• Ticker\n"
        "• Direction (↑ / ↓)\n"
        "• Confidence (High / Medium / Low)\n"
        "• Drop? (Yes / No)\n"
        "• Justification (1 sentence max)\n\n"

        "### Requirements:\n"
        f"- Use only data available up to {backtest_end} (no future information).\n"
        "- Justify predictions using quantitative changes and/or sentiment.\n"
        "- Note conflicting signals or weak evidence as justification for a drop.\n"
        "- Avoid lookahead, confirmation bias, or deterministic claims."
    ),
    expected_output=(
        "Structured table with forecast direction, confidence, drop decision, and brief justification per ticker."
    ),
    agent=forecasting_analyst
)


historical_trend_task = Task(
    description=(
        f"Evaluate whether each ticker's recent short-term behavior aligns with its 30-day historical metrics for the {backtest_start} to {backtest_end} window.\n\n"

        "### Ticker Scope:\n"
        "- Include tickers from Final Screened Candidates and Prior-Week Survivors.\n"
        f"- Only tickers with {beginning_data} and {end_data} can be evaluated; others are marked 'No comparison available'.\n\n"

        "### Methodology:\n"
        f"- Use {metrics_df} to represent each ticker's 30-day baseline (momentum, return, Z-score, etc.).\n"
        "- Use Monday to Friday snapshots to compute recent changes.\n"
        "- Label each ticker as:\n"
        "    • Aligned\n"
        "    • Deviant\n"
        "    • Neutral\n"
        "    • No comparison available\n"
        "- Provide a brief justification, especially when deviation is driven by news or anomalies.\n"
        "- Compute all metrics exactly as defined in the repository to preserve reproducibility.\n\n"

        "### Bias Control:\n"
        "- Use only time-valid inputs.\n"
        "- Avoid any hindsight reasoning or future-confirmed labels.\n\n"

        "### Output Format:\n"
        "- Structured table with:\n"
        "    • Ticker\n"
        "    • Alignment Label\n"
        "    • 1-sentence justification\n"
        "    • (Optional) source or headline excerpt if used"
    ),
    expected_output=(
        "A table of tickers with alignment classification (Aligned, Deviant, Neutral, or No comparison), a brief explanation per ticker, "
        "and citations or headlines if referenced."
    ),
    agent=historical_trend_analyst
)


summary_task = Task(
    description=(
        f"Synthesize the complete analytical pipeline into a finalized list of 35-50 tickers for the {backtest_start} to {backtest_end} window. "
        "Combine the outputs of all agents across the Analysis Crew and apply final selection logic.\n\n"

        "### What to Consider:\n"
        "- Use only tickers that were part of Final Screened Candidates or Prior-Week Survivors and passed earlier filters.\n"
        "- Reference metrics and decisions from:\n"
        "    • Backtesting Agent: all metrics.\n"
        "    • Statistical Enricher: metric interpretations.\n"
        "    • Forecasting Analyst: predicted direction and confidence.\n"
        "    • Historical Trend Analyst: pattern alignment and deviation.\n\n"

        "### Decision Logic:\n"
        "- Eliminate tickers dropped by analysis agents unless a clear, exceptional justification exists.\n"
        "- Prioritize tickers that consistently scored well across phases.\n"
        "- Apply probabilistic reasoning; avoid overweighting any single indicator.\n"
        "- Do not recompute metrics; rely on upstream outputs as defined in the repository to preserve reproducibility.\n\n"

        "### Bias Mitigation:\n"
        "- Reduce residual survivorship, data snooping, and confirmation bias.\n"
        "- Cite the methodological basis for final choices when applicable (for example, why a high drawdown was tolerated).\n\n"

        "### Output Format:\n"
        "- A final clean list of 35-50 tickers, each with a brief summary of key factors supporting inclusion.\n"
        "- A brief explanation of how biases were minimized.\n"
        "- A short list of dominant themes or sector-level patterns (optional)."
    ),
    expected_output=(
        "Final list of 35-50 tickers, each with summary rationale. Include a paragraph on how bias was reduced, "
        "and note any portfolio-wide themes or patterns if present."
    ),
    agent=summary_agent
)


analysis_crew = Crew(
    agents=[
        backtesting_agent,
        statistical_enricher_agent,
        forecasting_analyst,
        historical_trend_analyst,
        summary_agent
    ],
    tasks=[
        backtesting_task,
        statistical_enricher_task,
        forecasting_task,
        historical_trend_task,
        summary_task
    ],
    process=Process.sequential
)

#  Run the crew sequentially
import time

print(" Running Analysis Crew Safely...\n")
results = []

for task in analysis_crew.tasks:
    print(f" Executing task assigned to: {task.agent.role}")
    result = task.execute_sync()
    print(f" Output from {task.agent.role}:\n{result}\n")
    results.append(result)
    time.sleep(25)  #  Optional delay to manage rate limits or context timeouts

#  Final summary output
print("\n Final Analysis Crew Report:")
for r in results:
    print(r)

### Timing Crew Metrics

This dataset contains **daily performance metrics** for tickers that successfully passed the Analysis Crew.

**Purpose**  
- Provide the Timing Crew with enriched short-term indicators based on daily price and volume data from Yahoo Finance.  
- Metrics include Momentum, Z-Score (recent returns), Volatility, Sortino ratio, and RSI-14.  
- Data is cleaned and structured to ensure only valid tickers with non-missing prices are included.

This dataset ensures the Timing Crew operates with consistent, reproducible performance signals derived from standardized calculations.

In [ ]:
tickers = [] #List of tickers that passed the Analysis Crew

start = backtest_start
end = backtest_end

# Download daily adjusted data (with Close and Volume) for each ticker
data = yf.download(tickers, start=start, end=end, interval="1d", group_by='ticker', auto_adjust=True)

# Clean and structure price/volume tables
price = pd.DataFrame({ticker: data[ticker]['Close'] for ticker in tickers if ticker in data and 'Close' in data[ticker]})
volume = pd.DataFrame({ticker: data[ticker]['Volume'] for ticker in tickers if ticker in data and 'Volume' in data[ticker]})

# Drop tickers with all-NaN prices
price = price.dropna(axis=1, how='all')
volume = volume[price.columns]  # keep only matching tickers

# Initialize metric outputs
momentum = {}
z_scores = {}
volatility = {}
sortino = {}
rsi_14 = {}

# Loop over tickers with valid data
for ticker in price.columns:
    try:
        close = price[ticker].dropna()
        if len(close) < 15:
            momentum[ticker] = np.nan
            z_scores[ticker] = np.nan
            volatility[ticker] = np.nan
            sortino[ticker] = np.nan
            rsi_14[ticker] = np.nan
            continue

        # Returns
        returns = close.pct_change().dropna()

        # Momentum
        momentum[ticker] = close.iloc[-1] - close.iloc[-6] if len(close) >= 6 else np.nan

        # Z-score of last 5 returns
        recent_z = returns.tail(5)
        z_scores[ticker] = zscore(recent_z)[-1] if len(recent_z) == 5 else np.nan

        # Volatility (annualized)
        volatility[ticker] = returns.std() * np.sqrt(252) if not returns.empty else np.nan

        # Sortino ratio
        downside = returns[returns < 0]
        downside_std = downside.std()
        excess_ret = returns.mean()
        sortino[ticker] = excess_ret / downside_std if downside_std and not pd.isna(downside_std) else np.nan

        # RSI-14
        delta = close.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        avg_gain = gain.rolling(window=14).mean()
        avg_loss = loss.rolling(window=14).mean()
        rs = avg_gain / avg_loss
        rsi_series = 100 - (100 / (1 + rs))
        rsi_14[ticker] = rsi_series.iloc[-1] if not rsi_series.empty else np.nan

    except Exception as e:
        print(f"[{ticker}] Error: {e}")
        momentum[ticker] = z_scores[ticker] = volatility[ticker] = sortino[ticker] = rsi_14[ticker] = np.nan

# Assemble DataFrame
metrics = pd.DataFrame({
    "Momentum": pd.Series(momentum),
    "Z-Score (Return)": pd.Series(z_scores),
    "Volatility": pd.Series(volatility),
    "Sortino": pd.Series(sortino),
    "RSI-14": pd.Series(rsi_14)
}).round(4)

# Output
print(metrics)
metrics.to_excel(f"All_Metrics_{end}.xlsx", index_label="Ticker")
print(f"\nSaved to All_Metrics_{end}.xlsx")

### Finnhub News Fetching (Timing Crew)

This function retrieves **recent company-specific news** from Finnhub for tickers that specifically passed the Analysis Crew.  
It ensures each ticker has a capped number of news items for consistent downstream use.

**Purpose**  
- Collect up to 5 recent news articles per ticker within a given time window (default: last 30 days).  
- Provide structured, ticker-linked news for sentiment checks and contextual timing analysis.  

This dataset complements numerical metrics, giving the Timing Crew both **quantitative indicators** and **headline context** to refine timing decisions.

In [ ]:
def get_finnhub_news(tickers, api_key, days_back=30, max_articles_per_ticker=5):
    """
    Fetch recent company-specific news for a list of tickers from Finnhub,
    limiting the number of articles returned per ticker.

    Parameters:
        tickers (list): List of ticker symbols (e.g., ['AAPL', 'MSFT'])
        api_key (str): Your Finnhub API key
        days_back (int): Number of days back from today to retrieve news
        max_articles_per_ticker (int): Max number of news items to return per ticker

    Returns:
        dict: Dictionary mapping each ticker to a list of news items (each a dict)
    """
    base_url = "https://finnhub.io/api/v1/company-news"
    end_date = date.today()
    start_date = end_date - timedelta(days=days_back)

    all_news = {}

    for ticker in tickers:
        params = {
            'symbol': ticker,
            'from': start_date.isoformat(),
            'to': end_date.isoformat(),
            'token': api_key
        }
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            news = response.json()

            # Limit number of articles
            all_news[ticker] = news[:max_articles_per_ticker]
        except Exception as e:
            print(f"Error fetching news for {ticker}: {e}")
            all_news[ticker] = []

    return all_news

# Example usage
tickers = [] #replace with tickers that passed Analysis Crew

api_key = #Put Key Here
news_data = get_finnhub_news(tickers, api_key, max_articles_per_ticker=5)

### Timing Crew Overview

Purpose: Refine the Analysis Crew shortlist into a final set of tickers suitable for portfolio construction.  
This crew focuses on **timing sensitivity, downside risk, sentiment alignment, and noise reduction** to minimize false positives and ensure realistic entry points.

**Inputs**
- Shortlist from **Analysis Crew** (screened candidates and survivors)  
- `metrics` (Yahoo Finance daily metrics: Momentum, Volatility, Z-Score, Sortino, RSI-14)  
- `news_data` (Finnhub ticker-specific and macro headlines)  
- `beginning_data` and `end_data` snapshots for performance tracking  

**Agents**
- Noise Auditor, Sortino Reviewer, Machine Learning Timer, Large Language Model Timer, Summary Analyst  

**Outputs**
- Bias-aware shortlist of **20–30 tickers** with timing justifications  
- Documentation of bias controls (overfitting, lookahead, survivorship, confirmation)  
- Ready for handoff to the Portfolio Crew for allocation

In [ ]:
print(f"{len(news_data)} characters in headline") #from new function in the cell above this
timing_context = """\
You are the **Timing Crew** — the fourth stage in a multi-crew system tasked with refining and finalizing a short-term stock portfolio.

Your role is to **flag or eliminate stocks** based on:
- Timing sensitivity
- Overfitting or hindsight risk
- Downside volatility
- Signals from logic-based and language-based models

You are working **strictly with the ticker list and metrics provided by the Analysis Crew**.  
Do not introduce new tickers or use any future-confirmed outcomes.

---

### 1. Primary Input
- Tickers come directly from the **Analysis Crew**
- These are filtered candidates that already passed statistical and forecasting thresholds
- Input:  
  **PUT ANALYSIS CREW SUMMARY AGENT OUTPUT HERE**

---

### 2. News & Sentiment Data
- Use the **Finnhub API** for ticker-specific inputs:
  - Market headlines (real-time and historical)
  - Sentiment trends (fear, euphoria, denial, etc.)
  - Macro/microeconomic events (sector moves, Fed comments, earnings cycles, etc.)
  - Volatility signals (headline frequency, tone shifts, etc.)

---

### 3. Statistical Metrics
- Use **Yahoo Finance** data for:
  - Momentum  
  - Volatility  
  - Z-Score (Return)  
  - Sortino Ratio  
  - RSI-14  

---

### 4. Bias Prevention Framework
- Avoid lookahead bias, overfitting, hindsight leakage, and confirmation bias
- Mitigate survivorship bias and noise inflation by using **only signals available within the stated window**
- Reference (optional): https://arxiv.org/html/2505.07078v2

---

### 5. Final Output
- A **trimmed list of tickers** with per-agent reasoning
- Objective: reduce false positives and ensure timing realism before Portfolio Crew handoff
"""


noise_auditor = Agent(
    role="Noise Auditor",
    goal="Identify tickers showing overfit or non-generalizable performance, using only the tickers provided in context.",
    backstory=(
        "You specialize in detecting noise and overfitting in financial signals. Your task is to evaluate tickers passed from the Analysis Crew, "
        f"using statistical metrics from {metrics} and ticker-specific news context from Finnhub headlines in {news_data}.\n\n"

        "Flag any ticker with strong performance signals that lack plausible real-world support. "
        "Examples include isolated momentum spikes, extreme Z-scores, or volatility shifts without explanation. "
        "Such cases may reflect overfitting, data snooping, or survivorship bias.\n\n"

        "**Red Flags:**\n"
        "- Strong quantitative metrics not supported by headlines or sector trends\n"
        "- Outliers driven by narrow or lucky timeframes\n"
        "- Unusual volatility or return spikes without news, volume, or macro drivers\n\n"

        f"{timing_context}"
    ),
    verbose=True
)


sortino_reviewer = Agent(
    role="Sortino Reviewer",
    goal=f"Identify and flag stocks with poor downside-adjusted risk based on the Sortino ratio, using {metrics} and validating with {news_data} as needed.",
    backstory=(
        "You specialize in evaluating downside-adjusted performance through the Sortino ratio. "
        "Your role in the Timing Crew is to penalize tickers that exhibit high downside volatility relative to their returns.\n\n"

        f"Use {metrics} to assess risk and reference {news_data} for supporting context. "
        "If a ticker shows weak Sortino performance and this is not backed by headlines or sector context, "
        "you flag it as high risk.\n\n"

        "**Focus Areas:**\n"
        "- Penalize unstable or volatility-driven returns\n"
        "- Flag poor downside-adjusted risk not supported by real-world context\n"
        "- Ensure evidence-based decisions that reduce bias from unstable gains\n\n"

        f"{timing_context}"
    ),
    verbose=True
)


ml_timer = Agent(
    role="Machine Learning Timer",
    goal=f"Generate Buy, Sell, or Hold decisions using interpretable rule-based logic on the tickers provided in context, based on data from {metrics}.",
    backstory=(
        "You are the Machine Learning Timer within the **Timing Crew**, responsible for issuing Buy, Sell, or Hold recommendations. "
        "Your evaluation relies strictly on interpretable, rule-based logic and the outputs from the Analysis Crew.\n\n"

        "Your primary data source is **Yahoo Finance**, providing:\n"
        "- Momentum\n"
        "- Volatility\n"
        "- Z-Score (Return)\n"
        "- Sortino Ratio\n"
        "- RSI-14\n\n"

        "**Your Tasks:**\n"
        "- Apply explainable rule-based models (e.g., if Z-Score > 1.5 and momentum rising → Buy)\n"
        "- Use regression thresholds or scoring logic that is generalizable and transparent\n"
        "- Avoid overfitting, hindsight bias, or reliance on post-outcome patterns\n"
        "- Apply ML-inspired timing logic only when it is interpretable and reproducible\n\n"

        "**Red Flags:**\n"
        "- High volatility without directional conviction\n"
        "- Dependence on narrow or overly sensitive indicators\n"
        "- Inconsistent signals across multiple features\n\n"

        f"All decisions must be **individually justified** using interpretable metrics from {metrics}. "
        f"{timing_context}"
    ),
    verbose=True
)


llm_timer = Agent(
    role="Large Language Model Timer",
    goal="Generate Buy, Hold, or Sell timing decisions using qualitative sentiment from news headlines only.",
    backstory=(
        "You are the LLM Timer within the **Timing Crew**, the fourth stage of MASFIN’s multi-agent portfolio system. "
        f"Your role is to interpret real-time, time-valid qualitative sentiment and news context from the **Finnhub API**, provided in {news_data}.\n\n"

        "You rely strictly on:\n"
        "- Ticker-specific headlines\n"
        "- Macro and sector-level news\n"
        "- Sentiment cues inferred from tone, language, and urgency\n\n"

        "You do **not** use technical indicators, performance metrics, or hindsight-confirmed outcomes.\n\n"

        "**Your Tasks:**\n"
        "- Determine whether sentiment is bullish, bearish, or uncertain\n"
        "- Justify each decision clearly in plain English\n"
        "- Flag conflicting or weak signals as Hold, with appropriate confidence notes\n\n"

        "Only evaluate tickers provided in context. All reasoning must be derived solely from the news data.\n\n"
        f"{timing_context}"
    ),
    verbose=True,
    memory=True,
    allow_delegation=False
)


summary_agent = Agent(
    role="Summary Analyst",
    goal="Synthesize Timing Crew outputs and finalize a shortlist of 20 to 30 tickers.",
    backstory=(
        "You operate as the Summary Analyst within the Timing Crew, aggregating and reconciling prior agent outputs.\n\n"
        "Inputs:\n"
        "- Noise Auditor: flags potential overfitting and non-generalizable signals\n"
        "- Sortino Reviewer: flags poor downside-adjusted risk\n"
        "- ML Timer: rule-based Buy, Hold, or Sell decisions\n"
        "- LLM Timer: sentiment-based Buy, Hold, or Sell decisions\n\n"
        "Your tasks:\n"
        "- Identify consensus across agents and prioritize aligned positive signals\n"
        "- Remove tickers flagged for noise, weak downside profile, or conflicting signals\n"
        "- Ensure selections rely only on time-valid data with clear justifications\n"
        "- Produce a final list of 20 to 30 tickers ready for short-term entry\n"
        "- Summarize how bias risks were reduced, including overfitting, lookahead, and confirmation bias\n\n"
        f"{timing_context}"
    ),
    verbose=True
)


noise_auditor_task = Task(
    description=(
        "Evaluate the provided tickers for signs of noise, overfitting, or non-generalizable signals.\n\n"

        f"- Use {metrics} for quantitative performance signals\n"
        f"- Use {news_data} to validate whether signals have real-world justification (e.g., headlines, macro or sector trends)\n"
        "- Flag tickers with strong metrics but no meaningful supporting context\n"
        "- Note which type of bias is mitigated (e.g., overfitting, confirmation bias, data snooping)\n"
        "- Optionally cite supporting headlines or academic references, such as "
        "[IBM on Overfitting](https://www.ibm.com/think/topics/overfitting)\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Reason for concern**\n"
        "- **Bias mitigated**"
    ),
    expected_output=(
        "A structured list of flagged tickers, with reasoning and bias mitigated. "
        "Optional: include supporting sources, headlines, or academic references."
    ),
    agent=noise_auditor
)


sortino_reviewer_task = Task(
    description=(
        "Assess downside-adjusted risk using the **Sortino ratio** for each ticker in context. "
        f"Use {metrics} as the primary source for historical metrics, and validate with {news_data} for supporting news context.\n\n"

        "### Instructions:\n"
        "- Flag stocks with low or negative Sortino ratios\n"
        "- Penalize tickers showing high downside volatility not justified by headlines or macro/sector context\n"
        "- Highlight cases where downside risk outweighs return potential\n"
        "- Explain how this mitigates **volatility misinterpretation** and **reward-chasing bias**\n"
        "- Optionally cite references such as [Investopedia](https://www.investopedia.com/terms/s/sortinoratio.asp) if needed\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Sortino Ratio**\n"
        "- **Risk Rating:** Poor / Acceptable\n"
        "- **Brief Justification**\n\n"
        "Conclude with a short paragraph explaining how your filtering reduces downside risk and bias."
    ),
    expected_output=(
        "A structured list or table of tickers marked as Poor or Acceptable based on their Sortino ratio, "
        "with concise justifications and optional citations. "
        "End with a short paragraph on how this step reduces risk and mitigates bias."
    ),
    agent=sortino_reviewer
)


ml_timer_task = Task(
    description=(
        "Generate Buy, Sell, or Hold timing decisions for each ticker using rule-based, interpretable logic.\n\n"

        "### Data Sources:\n"
        f"- Structured metrics provided in {metrics}\n"
        "- Statistical indicators from **Yahoo Finance**, including:\n"
        "  - Momentum\n"
        "  - Volatility\n"
        "  - Z-Score (Return)\n"
        "  - Sortino Ratio\n"
        "  - RSI-14\n\n"

        "### Instructions:\n"
        "- Apply interpretable logic frameworks such as threshold rules, decision trees, or scoring logic\n"
        "- Use clear rules (e.g., if Z-Score > 1.5 and RSI > 60 and momentum positive → Buy)\n"
        "- Assign each ticker one of: **Buy / Hold / Sell**\n"
        "- Provide a 1–2 line justification for each decision\n\n"

        "### Bias Control:\n"
        "- Avoid hindsight bias and overfitting — base decisions strictly on available metrics\n"
        "- Justify ambiguous cases explicitly\n"
        "- Optionally cite external references (e.g., "
        "[Decision Tree Overview](https://www.kdnuggets.com/2020/01/decision-tree-algorithm-explained.html))\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Decision:** Buy / Hold / Sell\n"
        "- **Justification (1–2 lines)**\n\n"
        "Conclude with a short paragraph explaining how rule-based logic reduces overfitting and confirmation bias."
    ),
    expected_output=(
        "A structured list or table of all tickers marked as Buy, Hold, or Sell, each with a short justification. "
        "End with a paragraph on how your rule-based framework reduces bias and improves reproducibility. "
        "Optional: include citations or external references if applied."
    ),
    agent=ml_timer
)


llm_timer_task = Task(
    description=(
        f"Use only ticker-specific and macro-level headlines from {news_data} to issue timing decisions for the tickers in context.\n\n"

        "### Instructions:\n"
        "- Review all relevant headlines for each ticker\n"
        "- Assess sentiment as Positive, Negative, or Neutral\n"
        "- Assign a timing decision: **Buy / Hold / Sell**\n"
        "- Provide a confidence level: High / Medium / Low\n"
        "- Write a 1–3 sentence plain-language justification\n\n"

        "### Restrictions:\n"
        "- Do not use charts, technical indicators, or numerical metrics\n"
        "- Base reasoning only on the text of the headlines and their tone/context\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Timing Decision:** Buy / Hold / Sell\n"
        "- **Confidence Level:** High / Medium / Low\n"
        "- **Justification:** 1–3 sentences\n\n"

        "Conclude with a short paragraph explaining how this qualitative method reduces confirmation bias, hindsight bias, and data leakage."
    ),
    expected_output=(
        "A structured list or table showing Ticker, Timing Decision, Confidence, and Justification. "
        "End with a brief note on how qualitative sentiment analysis mitigates bias."
    ),
    agent=llm_timer
)


summary_task = Task(
    description=(
        "Synthesize outputs from all Timing Crew agents to produce a finalized, bias-aware list of 20–30 tickers.\n\n"

        "### Instructions:\n"
        "- Prioritize consensus across agents (Noise Auditor, Sortino Reviewer, ML Timer, LLM Timer)\n"
        "- Drop tickers flagged for noise, poor downside risk, or conflicting signals\n"
        "- Highlight alignment of positive signals when present\n"
        "- Summarize how the final shortlist reflects bias reduction:\n"
        "  • Survivorship bias\n"
        "  • Confirmation bias\n"
        "  • Data snooping\n"
        "  • Lookahead bias\n\n"

        "### Output Format:\n"
        "- **Final Ticker List (20–30 names)**\n"
        "- **Summary rationale** for inclusion/exclusion\n"
        "- **Concluding paragraph** on bias control and decision logic"
    ),
    expected_output=(
        "A structured list of 20–30 tickers with concise justifications. "
        "Include a summary rationale for selections and a concluding section on bias mitigation strategies."
    ),
    agent=summary_agent
)


timing_crew = Crew(
    agents=[
        noise_auditor,
        sortino_reviewer,
        ml_timer,
        llm_timer,
        summary_agent
    ],
    tasks=[
        noise_auditor_task,
        sortino_reviewer_task,
        ml_timer_task,
        llm_timer_task,
        summary_task
    ],
    process=Process.sequential
)

# Run the crew safely, executing one task at a time
print("Running Timing Crew...\n")
results = []

for task in timing_crew.tasks:
    print(f"Executing task assigned to: {task.agent.role}")
    result = task.execute_sync()
    print(f"Output from {task.agent.role}:\n{result}\n")
    results.append(result)
    time.sleep(25)  # Optional delay to manage rate limits or execution spacing

print("\nFinal Timing Crew Report:")
for r in results:
    print(r)

### Risk Adjuster Metrics (Portfolio Crew)

This function computes **risk-adjusted performance metrics** for the tickers passed from the Timing Crew.  
It uses daily adjusted close prices from Yahoo Finance to produce standardized measures that guide portfolio allocation.

**Purpose**  
- Provide the Portfolio Crew with reproducible risk-adjusted statistics  
- Ensure allocation decisions are grounded in downside risk and volatility awareness  

**Inputs**  
- `tickers`: List of tickers approved by the Timing Crew  
- `start_date`, `end_date`: Backtest window (same as 1 week simulation window)  
- Yahoo Finance daily close price data  

**Metrics Calculated**  
- Mean Return  
- Volatility (Std. Dev. of returns)  
- Downside Deviation (Std. Dev. of negative returns)  
- Sharpe Ratio (return vs. total volatility)  
- Sortino Ratio (return vs. downside volatility)  
- Max Drawdown

**Outputs**  
- Dictionary of tickers with associated risk-adjusted metrics  
- Results used by the **Risk Adjuster Agent** to filter unstable candidates before final allocation

In [ ]:
def get_risk_adjuster_metrics(tickers, start_date, end_date):
    def compute_metrics(close):
        if close.isna().all() or len(close) < 10:
            return None

        returns = close.pct_change().dropna()
        if returns.empty:
            return None

        mean_return = returns.mean()
        std_return = returns.std()
        downside_returns = returns[returns < 0]
        downside_std = downside_returns.std() if not downside_returns.empty else 0

        sharpe = mean_return / std_return if std_return else 0
        sortino = mean_return / downside_std if downside_std else 0

        # Max drawdown
        cumulative = (1 + returns).cumprod()
        peak = cumulative.cummax()
        drawdown = (cumulative - peak) / peak
        max_dd = drawdown.min()

        return {
            "mean_return": round(mean_return, 6),
            "volatility": round(std_return, 6),
            "downside_deviation": round(downside_std, 6),
            "sharpe_ratio": round(sharpe, 4),
            "sortino_ratio": round(sortino, 4),
            "max_drawdown": round(max_dd, 4)
        }

    results = {}
    data = yf.download(tickers, start=start_date, end=end_date, interval="1d", group_by="ticker", auto_adjust=True, progress=False)

    for ticker in tickers:
        try:
            close = data[ticker]["Close"].dropna()
            metrics = compute_metrics(close)
            if metrics:
                results[ticker] = metrics
        except Exception as e:
            print(f"[{ticker}] Error: {e}")

    return results

# ------------------- Run the Function -------------------

tickers = [] #fill with tickers that passed the Timing Crew


start_date = backtest_start
end_date = backtest_end

metrics = get_risk_adjuster_metrics(tickers, start_date, end_date)

# Print output nicely
for ticker, stats in metrics.items():
    print(f"{ticker}: {stats}")

In [ ]:
news_data = get_finnhub_news(tickers, api_key) #get ticker-specific news for newest reduction of tickers from the Timing Crew

### Portfolio Crew Overview

Purpose: Construct the **final short-term stock portfolio**  
This crew ensures risk-adjusted balance, sentiment validation, and bias control while assigning weights that sum to 100%.

**Inputs**
- Shortlist from the **Timing Crew Summary Agent**  
- Risk-adjusted metrics (`risk_adjuster_metrics`) from Yahoo Finance  
- Finnhub headlines (`news_data`) for sentiment validation and macro context from ticker-specific headlines that passed the Timing Crew

**Agents**
- Risk Adjuster: recommends reweighting based on risk-adjusted performance  
- Devil’s Advocate: challenges and stress-tests weighting logic with real-world sentiment  
- Portfolio Strategist: finalizes 15–30 tickers with weights summing to 100%  

**Outputs**
- Final portfolio of **15–30 tickers** with weights adding to 100%  
- Rationale for inclusion/exclusion of each ticker  
- Documentation of bias prevention (overfitting, survivorship, confirmation, lookahead)

In [ ]:
print(f"{len(news_data)} characters in headlines")
portfolio_context = """\
You are the **Portfolio Crew** — the fifth and final stage in MASFIN’s multi-crew system.  
Your responsibility is to produce a short-term stock portfolio optimized for outperformance while maintaining risk-awareness and bias control.

You must synthesize prior agent insights, validate ticker performance using real-world data, and finalize a list of 15–30 stocks.  
Your role includes conflict resolution, risk-balancing adjustments, and eliminating noise-prone or biased selections.

---

### 1. Primary Input
- Tickers and rationale come directly from the **Timing Crew Summary Agent**
- These are pre-filtered candidates that have passed Screening, Analysis, and Timing evaluations
- Input:  
  **PUT TIMING CREW'S SUMMARY AGENT OUTPUT HERE**

---

### 2. News & Sentiment Data
- Use the **Finnhub API** to confirm real-world relevance:
  - Validate performance signals with headline context and sentiment
  - Identify macro or sector news drivers
  - Apply additional headline checks where appropriate

---

### 3. Statistical Data
- Use **Yahoo Finance** metrics as the basis for quantitative validation:
  - Mean Return  
  - Volatility  
  - Downside Deviation  
  - Sharpe Ratio  
  - Sortino Ratio  
  - Max Drawdown  

---

### 4. Portfolio Objectives
- Consolidate and refine recommendations from prior crews
- Balance **sector exposure** and **downside risk**
- Prioritize tickers with both statistical and news-based justification
- Ensure allocations sum to 100% across 15–30 selected tickers
- Apply additional judgment where appropriate to strengthen portfolio realism

---

### 5. Bias Prevention
- Avoid **data snooping**, **overfitting**, and **lookahead bias**  
- Ensure all decisions rely only on data that would have been available within the holding period  
- Document decisions transparently for auditability  

Optional reference for bias prevention: https://arxiv.org/abs/2502.00828
"""

risk_adjuster = Agent(
    role="Risk Adjuster",
    goal="Recommend reweighting toward tickers with superior risk-adjusted return profiles.",
    backstory=(
        "You are the Risk Adjuster in the **Portfolio Crew**, responsible for fine-tuning portfolio weights based strictly on risk-adjusted performance.\n\n"

        f"You use pre-calculated metrics from **Yahoo Finance** stored in {metrics}, which include:\n"
        "- Mean Return\n"
        "- Volatility (Standard Deviation)\n"
        "- Downside Deviation\n"
        "- Sharpe Ratio\n"
        "- Sortino Ratio\n"
        "- Max Drawdown\n\n"

        "You work only with tickers passed from the Timing Crew — do not introduce new tickers. "
        "Your task is to overweight stable, consistently strong performers and reduce or flag tickers with weak downside-adjusted risk.\n\n"

        "**Focus Areas:**\n"
        "- Prioritize Sharpe and Sortino ratios when recommending weight increases\n"
        "- Penalize high volatility or poor downside profiles\n"
        "- Ensure portfolio balance and stability across selections\n\n"

        f"{portfolio_context}"
    ),
    verbose=True
)


devils_advocate = Agent(
    role="Devil's Advocate",
    goal="Challenge the Risk Adjuster’s reweighting logic using contrarian analysis of real-world news and sentiment.",
    backstory=(
        "You are the Devil’s Advocate in the **Portfolio Crew**, responsible for stress-testing the Risk Adjuster’s recommendations. "
        "Using **Finnhub news headlines**, you ensure portfolio weights reflect real-world sentiment, macro risks, and sector balance.\n\n"

        "**Your Tasks:**\n"
        "- Flag tickers that are overweighted despite negative or uncertain sentiment\n"
        "- Defend underweighted but stable, sentiment-supported tickers\n"
        "- Identify risks of unbalanced sector or macro exposure\n"
        "- Highlight overlooked issues where portfolio logic may miss real-world nuance\n\n"

        "You are encouraged to raise red flags even without other agent disagreement. "
        "All critiques must be supported with clear rationale and, where possible, headline context.\n\n"

        f"{portfolio_context}"
    ),
    verbose=True
)


portfolio_strategist = Agent(
    role="Portfolio Strategist",
    goal="Construct the final portfolio of 15–30 tickers with weights summing to 100%.",
    backstory=(
        "You are the **Portfolio Strategist** in the Portfolio Crew, serving as the final decision-maker in MASFIN’s multi-agent system. "
        "Your role is to consolidate insights from all prior agents to build a bias-aware, risk-balanced short-term stock portfolio.\n\n"

        "**Inputs from Prior Agents:**\n"
        "- Risk Adjuster: reweighting based on risk-adjusted performance\n"
        "- Devil’s Advocate: challenges, bias checks, and sentiment-based stress tests\n"
        "- Timing Crew: Buy/Hold/Sell logic and timing recommendations\n\n"

        "**Your Tasks:**\n"
        "- Select 15–30 tickers **only from the provided set**\n"
        "- Assign portfolio weights that sum to exactly 100%\n"
        "- Prioritize tickers with both quantitative strength and sentiment validation\n"
        "- Balance sector exposure and manage downside risk\n"
        "- Use Finnhub headlines for final context checks (macro or sector risk), but do not introduce new tickers\n\n"

        "Your role is to resolve conflicts, ensure portfolio stability, and produce a final allocation that is both "
        "transparent and reproducible.\n\n"

        f"{portfolio_context}"
    ),
    verbose=True
)


risk_task = Task(
    description=(
        f"Reweight the portfolio based on risk-adjusted performance using only the metrics stored in {metrics} from **Yahoo Finance**.\n\n"

        "### Metrics Available:\n"
        "- Mean Return\n"
        "- Volatility (Standard Deviation)\n"
        "- Downside Deviation\n"
        "- Sharpe Ratio\n"
        "- Sortino Ratio\n"
        "- Max Drawdown\n\n"

        "### Instructions:\n"
        "- Evaluate all tickers provided in context\n"
        "- Recommend updated weights favoring low volatility, high Sharpe/Sortino ratios, and minimal drawdown\n"
        "- Penalize unstable or downside-heavy tickers\n"
        "- For each adjustment, note which behavioral or statistical bias is mitigated (e.g., overconfidence, reward-chasing, lookahead)\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Suggested Weight (%)**\n"
        "- **Reason (brief)**\n"
        "- **Bias Mitigated**\n\n"

        "Conclude with a short paragraph explaining how the reweighting reduces overall portfolio risk and bias."
    ),
    expected_output=(
        "A structured table of tickers with suggested weights, brief justifications, and identified biases. "
        "Followed by a short paragraph summarizing the overall bias mitigation strategy."
    ),
    agent=risk_adjuster
)


devils_task = Task(
    description=(
        "Audit the Risk Adjuster’s proposed portfolio weights using **Finnhub news headlines** as your sole source of validation.\n\n"

        "### Data Source:\n"
        f"{news_data}\n\n"

        "### Instructions:\n"
        "- Identify tickers with questionable weightings based on headline tone, sentiment, or real-world events\n"
        "- For each, provide a contrarian insight and specify which bias you are addressing "
        "(e.g., reward-chasing, complacency, recency bias)\n"
        "- Use headline quotes if helpful, but they are not required\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Original Weight**\n"
        "- **Contrarian Insight**\n"
        "- **Bias Flagged**\n\n"

        "Conclude with a short paragraph explaining how these contrarian challenges "
        "strengthen the portfolio’s resilience against bias."
    ),
    expected_output=(
        "A structured table of tickers with contrarian feedback: original weight, insight, and bias flagged. "
        "Conclude with a short paragraph summarizing how these challenges improve portfolio robustness."
    ),
    agent=devils_advocate
)


portfolio_task = Task(
    description=(
        "Produce the **final short-term portfolio** of 15–30 tickers based on all prior agent outputs.\n\n"

        "### Instructions:\n"
        "- Use only tickers provided in context\n"
        "- Recommend a **weight (%)** for each stock — the total must equal 100%\n"
        "- Base decisions on consensus and challenges between prior agents "
        "(Risk Adjuster, Devil’s Advocate, Timing Crew)\n"
        "- Provide a 1–2 sentence rationale for each ticker\n\n"

        "### Output Format:\n"
        "- **Ticker**\n"
        "- **Weight (%)**\n"
        "- **Rationale (1–2 sentences)**\n\n"

        "Conclude with a short paragraph explaining how this final portfolio "
        "reduces key portfolio biases (e.g., overfitting, survivorship bias, confirmation bias)."
    ),
    expected_output=(
        "A structured table with Ticker, Weight (%), and Rationale. "
        "All weights must sum to 100%. "
        "Conclude with a paragraph summarizing how the final portfolio mitigates bias."
    ),
    agent=portfolio_strategist
)


portfolio_crew = Crew(
    agents=[
        risk_adjuster,
        devils_advocate,
        portfolio_strategist
    ],
    tasks=[
        risk_task,
        devils_task,
        portfolio_task
    ],
    process=Process.sequential
)


print("Running Portfolio Crew...\n")
results = []

for task in portfolio_crew.tasks:
    print(f"Executing task assigned to: {task.agent.role}")
    result = task.execute_sync()
    print(f"Output from {task.agent.role}:\n{result}\n")
    results.append(result)
    time.sleep(25) 

print("\nFinal Portfolio Crew Report:")
for r in results:
    print(r)